In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
# Read PVDAQ metadata from Bronze Lakehouse
bronze_file="abfss://ac09d151-f58f-40f3-bf13-a61a3c24bcdb@onelake.dfs.fabric.microsoft.com/d6215562-2475-44dc-a961-90b9dda1a1c2/Files/pvdaq/pipeline_test/systems_20250729.csv"

In [ ]:
df=(
    spark.read
    .option("header","true")
    .option("inferschema","true")
    .csv(bronze_file)
)

print("total records:", df.count())
print("total columns:",len(df.columns))

display(df.limit(5))




In [ ]:
# Inspect column names and data types
df.printSchema()

In [ ]:
# Identify columns containing only null values
from pyspark.sql import functions as f

null_counts= df.select([
    f.count(f.col(f"`{c}`")).alias(c)
    for c in df.columns
])

display(null_counts)



In [ ]:
from pyspark.sql import functions as f

# Select the five extra columns
extra_columns=["_c26", "_c27", "_c28", "_c29", "_c30"]

# Count non-null values in each column
df.select([
    f.count(f.col(c)).alias(c)
    for c in extra_columns
]).show()



In [ ]:
# Remove the five empty columns
df_silver=df.drop("_c26", "_c27", "_c28", "_c29", "_c30")

print("Rows:", df_silver.count())
print("columns:", len(df_silver.columns))

In [ ]:
# Find duplicate system IDs
duplicates=(
   df_silver
   .groupby("system_id")
   .count()
   .filter(f.col("count")>1) 
)
display(duplicates)



In [ ]:
# Inspect the duplicated records
display(
    df_silver
    .filter(f.col("system_id").isin(1332,4901))
    .orderBy("system_id")
)

In [ ]:
# Compare duplicate records column by column

duplicates_rows = df_silver.filter(
    f.col("system_id").isin(1332, 4901)
)

for system_id in [1332, 4901]:

    rows = duplicates_rows.filter(
        f.col("system_id") == system_id
    ).collect()

    print(f"\nSystem ID: {system_id}")

    for column in df_silver.columns:

        value_1 = rows[0][column]
        value_2 = rows[1][column]

        if value_1 != value_2:
            print(column, ":", value_1, "|", value_2)

In [ ]:
# Flag system IDs with conflicting metadata

df_silver=df_silver.withColumn(
    "metadata_conflict_flag",
    f.when(
        f.col("system_id").isin(1332,4901),1
    ).otherwise(0)
)

# Verify the results
display(
    df_silver
    .groupby("metadata_conflict_flag")
    .count()
)

In [ ]:
# Convert string columns to timestamps
df_silver=(
    df_silver
    .withColumn(
        "first_timestamp",
        f.to_timestamp("first_timestamp")
    )
.withColumn(
    "last_timestamp",
    f.to_timestamp("last_timestamp")
)
)

# Verify the updated schema
df_silver.printSchema()

In [ ]:
display(
    df_silver.select(
        "system_id",
         "first_timestamp",
         "last_timestamp"
 ).limit(5)
)

In [ ]:
# Inspect original timestamp values before conversion

df.select(
    "system_id",
    "first_timestamp",
    "last_timestamp"
).show(10, truncate=False)

In [ ]:
# Rebuild from the original Bronze DataFrame
df_silver = df.drop("_c26", "_c27", "_c28", "_c29", "_c30")

# Convert timestamps using the source date format
df_silver = (
    df_silver
    .withColumn(
        "first_timestamp",
        f.to_timestamp("first_timestamp", "M/d/yyyy H:mm")
    )
    .withColumn(
        "last_timestamp",
        f.to_timestamp("last_timestamp", "M/d/yyyy H:mm")
    )
)

# Verify results
df_silver.select(
    "system_id",
    "first_timestamp",
    "last_timestamp"
).show(10, truncate=False)

In [ ]:
df_silver.select(
    f.count("*").alias("total_records"),
    f.count("first_timestamp").alias("parsed_first"),
    f.count("last_timestamp").alias("parsed_last")
).show()

In [ ]:
df_silver.filter(
    f.col("first_timestamp").isNull()|
    f.col("last_timestamp").isNull()
).select(
    "system_id",
    "first_timestamp",
    "last_timestamp"
).show(truncate=False)

In [ ]:
df.filter(
    f.col("system_id").isin(12423, 12494)
).select(
    "system_id",
    "first_timestamp",
    "last_timestamp"
).show(truncate=False)

In [ ]:
df_silver = df_silver.withColumn(
    "metadata_conflict_flag",
    f.when(
        f.col("system_id").isin(1332, 4901),
        1
    ).otherwise(0)
)

In [ ]:
df_silver.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("silver_pvdaq_systems")

In [ ]:
spark.sql("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT system_id) AS unique_systems,
    SUM(CASE WHEN metadata_conflict_flag = 1 THEN 1 ELSE 0 END)
        AS conflicting_records,
    SUM(CASE WHEN first_timestamp IS NULL THEN 1 ELSE 0 END)
        AS missing_first_timestamps,
    SUM(CASE WHEN last_timestamp IS NULL THEN 1 ELSE 0 END)
        AS missing_last_timestamps
FROM silver_pvdaq_systems
""").show()

In [ ]:
from pyspark.sql import functions as f

# 1. Read the saved Silver Delta table
df_systems=spark.table("silver_pvdaq_systems")

# 2. Detect duplicate system IDs dynamically
duplicate_ids=(
    df_systems
    .groupby("system_id")
    .count()
    .filter(f.col("count")>1)
    .select("system_id")
)

# 3. Quarantine every record belonging to a duplicate ID
df_quarantine = df_systems.join(
    duplicate_ids,
    on="system_id",
    how="left_semi"
)

# 4. Keep only records with non-duplicated IDs
df_clean=df_systems.join(
   duplicate_ids,
    on="system_id",
    how="left_anti"
)

# 5. Validate before saving
print("Clean records:", df_clean.count())
print("Quarantine records:", df_quarantine.count())
print("Unique clean systems:", df_clean.select("system_id").distinct().count()) 


In [ ]:
# 1. Save duplicate records to quarantine
df_quarantine.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("quarantine_pvdaq_systems")

# 2. Save clean records to Silver
df_clean.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("silver_pvdaq_systems")



In [ ]:
df_quarantine

In [ ]:
spark.sql("""
    select count(*) as total_records
    from quarantine_pvdaq_systems
    """).show()

In [ ]:
df_clean.write\
.format("delta")\
.mode("overwrite")\
.saveAsTable("silver_pvdaq_systems")

In [ ]:
spark.sql("""
SELECT
    'Silver' AS table_name,
    COUNT(*) AS total_records
FROM silver_pvdaq_systems

UNION ALL

SELECT
    'Quarantine' AS table_name,
    COUNT(*) AS total_records
FROM quarantine_pvdaq_systems
""").show()

In [ ]:
spark.sql("""
SELECT
    'Silver' AS table_name,
    COUNT(*) AS total_records
FROM silver_pvdaq_systems

UNION ALL

SELECT
    'Quarantine' AS table_name,
    COUNT(*) AS total_records
FROM quarantine_pvdaq_systems
""").show()

In [ ]:
spark.sql("""
SELECT
    system_id,
    system_public_name,
    dc_capacity_kW,
    years,
    number_records,
    dataset_size_mb,
    available_sensor_channels
FROM silver_pvdaq_systems
WHERE qa_status = 'pass'
  AND dc_capacity_kW > 0
  AND years >= 3
  AND dataset_size_mb BETWEEN 1 AND 500
  AND available_sensor_channels >= 3
ORDER BY number_records DESC
LIMIT 15
""").show(15, truncate=False)

In [ ]:
spark.sql("""
SELECT
    system_id,
    system_public_name,
    site_location,
    dc_capacity_kW,
    tracking,
    type,
    tilt,
    azimuth,
    available_sensor_channels,
    first_timestamp,
    last_timestamp
FROM silver_pvdaq_systems
WHERE system_id IN (1199, 2105)
""").show(truncate=False)

In [ ]:
import requests
import xml.etree.ElementTree as ET

bucket_url = "https://oedi-data-lake.s3.amazonaws.com"

params = {
    "list-type": "2",
    "prefix": "pvdaq/csv/",
    "delimiter": "/"
}

response = requests.get(
    bucket_url,
    params=params,
    timeout=30
)

response.raise_for_status()

root = ET.fromstring(response.content)

ns = {
    "s3": "http://s3.amazonaws.com/doc/2006-03-01/"
}

prefixes = [
    element.text
    for element in root.findall(
        ".//s3:CommonPrefixes/s3:Prefix",
        ns
    )
]

for prefix in prefixes[:30]:
    print(prefix)

In [ ]:
params["prefix"] = "pvdaq/csv/pvdata/"

response = requests.get(
    bucket_url,
    params=params,
    timeout=30
)

response.raise_for_status()

root = ET.fromstring(response.content)

folders = [
    element.text
    for element in root.findall(
        ".//s3:CommonPrefixes/s3:Prefix", ns
    )
]

files = [
    element.text
    for element in root.findall(
        ".//s3:Contents/s3:Key", ns
    )
]

print("Requested URL:", response.url)
print("Folders:", folders[:10])
print("Files:", files[:10])

In [ ]:
params["prefix"] = "pvdaq/csv/pvdata/system_id=1199/"

params.pop("delimiter", None)

response = requests.get(
    bucket_url,
    params=params,
    timeout=30
)

response.raise_for_status()

root = ET.fromstring(response.content)

files = [
    element.text
    for element in root.findall(
        ".//s3:Contents/s3:Key",
        ns
    )
]

print("Files found:", len(files))

for file in files[:20]:
    print(file)

In [ ]:
import pandas as pd

file_url=(
     "https://oedi-data-lake.s3.amazonaws.com/"
    "pvdaq/csv/pvdata/system_id=1199/"
    "year=2010/month=10/day=1/"
    "system_1199__date_2010_10_01.csv"
)

df_sample=pd.read_csv(file_url)

print("rows and columns" , df_sample.shape)

print("\ncolumn names:")
print(df_sample.columns.tolist())

print("\nfirst 5 records:")
display(df_sample.head())